# Phase 3: Baseline Model (TF-IDF + Logistic Regression)

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from portfolio_projekt.data import LABELS, THRESHOLD, load_english, load_german
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.metrics import precision_recall_curve, precision_score, recall_score

## English baseline: word-level TF-IDF
Binary target: toxicity >= 0.5. Trained on the enriched training set, evaluated on
the unmodified validation split.

In [ ]:
df_en_train, df_en_val, df_en_test = load_english()
df_de_train, df_de_val, df_de_test = load_german()

for name, df in [("EN train", df_en_train), ("EN val", df_en_val), ("EN test", df_en_test),
                 ("DE train", df_de_train), ("DE val", df_de_val), ("DE test", df_de_test)]:
    print(f"{name:9} {df.shape}")

In [ ]:
y_en_train = df_en_train["toxicity"] >= THRESHOLD
y_en_val = df_en_val["toxicity"] >= THRESHOLD

vectorizer_word = TfidfVectorizer(min_df=20, sublinear_tf=True)
X_en_train = vectorizer_word.fit_transform(df_en_train["text"])
X_en_val = vectorizer_word.transform(df_en_val["text"])

model_word = LogisticRegression(max_iter=1000)
model_word.fit(X_en_train, y_en_train)
p_en_val = model_word.predict_proba(X_en_val)[:, 1]

print(f"Vocabulary size: {X_en_train.shape[1]}")
print(f"ROC-AUC: {roc_auc_score(y_en_val, p_en_val):.3f}")
print(f"PR-AUC:  {average_precision_score(y_en_val, p_en_val):.3f}")
print(f"F1 at threshold 0.5: {f1_score(y_en_val, p_en_val >= 0.5):.3f}")

In [ ]:
# Precision and recall at the default threshold of 0.5
print(f"At 0.5:  precision {precision_score(y_en_val, p_en_val >= 0.5):.3f}, "
      f"recall {recall_score(y_en_val, p_en_val >= 0.5):.3f}")

# Find the threshold with the best F1 score on the validation data
precision, recall, thresholds = precision_recall_curve(y_en_val, p_en_val)
f1 = 2 * precision * recall / (precision + recall + 1e-12)
best = f1[:-1].argmax()

print(f"Best threshold: {thresholds[best]:.2f}")
print(f"  F1 {f1[best]:.3f}, precision {precision[best]:.3f}, recall {recall[best]:.3f}")

## Adding character n-grams
Character n-grams (2–5) capture obfuscated spellings like "a$$" or "Sch***ß".

In [ ]:
def evaluate(y_true, p, name):
    """Return ROC-AUC, PR-AUC and the best F1 score with its threshold."""
    precision, recall, thresholds = precision_recall_curve(y_true, p)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    best = f1[:-1].argmax()
    return {
        "model": name,
        "roc_auc": roc_auc_score(y_true, p),
        "pr_auc": average_precision_score(y_true, p),
        "best_threshold": thresholds[best],
        "best_f1": f1[best],
    }

results_en = [evaluate(y_en_val, p_en_val, "word")]

In [ ]:
vectorizer_char = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(2, 5), min_df=20, sublinear_tf=True, max_features=200_000
)
X_en_train_char = vectorizer_char.fit_transform(df_en_train["text"])
X_en_val_char = vectorizer_char.transform(df_en_val["text"])

# Put word and character features side by side
X_en_train_both = hstack([X_en_train, X_en_train_char]).tocsr()
X_en_val_both = hstack([X_en_val, X_en_val_char]).tocsr()

model_both = LogisticRegression(max_iter=1000)
model_both.fit(X_en_train_both, y_en_train)
p_en_val_both = model_both.predict_proba(X_en_val_both)[:, 1]

results_en.append(evaluate(y_en_val, p_en_val_both, "word + char"))
pd.DataFrame(results_en).round(3)

In [ ]:
examples = [
    "You are an idiot",
    "You are an id10t",
    "You are such an A$$",
    "What an a s s",
    "Thanks for the article",
]

df_examples = pd.DataFrame({"text": examples})
X_word = vectorizer_word.transform(df_examples["text"])
X_both = hstack([X_word, vectorizer_char.transform(df_examples["text"])]).tocsr()

df_examples["p_word"] = model_word.predict_proba(X_word)[:, 1].round(2)
df_examples["p_word_char"] = model_both.predict_proba(X_both)[:, 1].round(2)
df_examples

## German baseline: word + character TF-IDF
Same method as for English. Evaluated overall and per source, because the base rates
differ strongly (RP 4.7 %, GermEval 35 %).

**Finding:** Adding character n-grams (2–5) improves all metrics
(PR-AUC 0.638 → 0.664, best F1 0.598 → 0.615). The best threshold is below 0.5
(0.38): even with enrichment, only 18 % of the training data is toxic, so the model
is conservative. The threshold is a design choice: low for a review queue
(fewer missed comments), high for automatic actions (fewer false alarms).

In [ ]:
def train_word_char(train_texts, y_train, val_texts):
    """Train word + character TF-IDF with logistic regression, return validation probabilities."""
    vec_word = TfidfVectorizer(min_df=20, sublinear_tf=True)
    vec_char = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(2, 5), min_df=20, sublinear_tf=True, max_features=200_000
    )
    X_train = hstack([vec_word.fit_transform(train_texts), vec_char.fit_transform(train_texts)]).tocsr()
    X_val = hstack([vec_word.transform(val_texts), vec_char.transform(val_texts)]).tocsr()

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    return model.predict_proba(X_val)[:, 1], (vec_word, vec_char, model)

In [ ]:
y_de_train = df_de_train["toxicity"] >= THRESHOLD
y_de_val = df_de_val["toxicity"] >= THRESHOLD

p_de_val, pipeline_de = train_word_char(df_de_train["text"], y_de_train, df_de_val["text"])

results_de = [evaluate(y_de_val, p_de_val, "DE all")]
for source in ["rp_crowd", "germeval2021"]:
    in_source = (df_de_val["source"] == source).values
    results_de.append(evaluate(y_de_val[in_source], p_de_val[in_source], f"DE {source}"))
    print(f"Base rate {source}: {y_de_val[in_source].mean():.1%}")

pd.DataFrame(results_de).round(3)

**Findings (German baseline):** Much weaker than English (ROC-AUC 0.818 vs. 0.935,
PR-AUC 0.250 vs. 0.664). Relative to the base rate, PR-AUC is 4.6x random for RP but
only 1.7x for GermEval. The model mainly learned RP (67k vs. 2.9k training comments) and
assigns very low probabilities to GermEval comments (best threshold 0.05). A single
threshold across sources would fail (EN 0.38, RP 0.14, GermEval 0.05) → thresholds per
language and source in Phase 6. German needs more data (Phase 4) and transfer from
English (Phase 5).